# PFAS Groundwater — T2 Multilabel Baseline (Colab)

**Task T2**: predict which individual PFAS compounds exceed their per-compound threshold (multilabel).

This notebook is **AUTONOMOUS** (CLAUDE.md §4): it bootstraps `src/` and the dataset via
`git clone` (no Google Drive required), then runs 5 models: Prevalence floor, Binary
Relevance, Masked Classifier Chain, Ensemble Classifier Chains, and **FrequencyClassChain**
(Dong et al. 2024-style cascade).

**Strict predictive mode**: no PFAS measurement is used as a feature (CLAUDE.md §1).

---
**PARAMETERS TO SET (cell below)**:
- `SMOKE_TEST` : `True` for a fast CPU sanity check (~3 min), `False` for the full GPU run.
- `REPO_URL` : GitHub repo URL (default `https://github.com/dnwiloic/pfas-gnn.git`).
- `GIT_REF` : branch or full commit SHA to pin (default `main`).
- `DATA_PATH` : path to the parquet, relative to the repo root (default `data/CA-PFAS-ASGWS.parquet`).

In [1]:
# ============================================================
#  USER PARAMETERS — edit these before running
# ============================================================

SMOKE_TEST = False          # True = fast CPU sanity check; False = full GPU run

# --- Repository (code + data) ---
# The dataset is versioned in the repo at data/CA-PFAS-ASGWS.parquet.
# git clone brings both src/ and data/ automatically. No Drive needed.
REPO_URL = "https://github.com/dnwiloic/pfas-gnn.git"   # <-- SET THIS if forked
GIT_REF  = "main"          # branch name or full commit SHA

# Path to the parquet file RELATIVE to the cloned repo root.
# Change only if the file was moved inside the repo.
DATA_PATH = "data/CA-PFAS-ASGWS.parquet"

print(f"SMOKE_TEST={SMOKE_TEST}  REPO_URL={REPO_URL!r}  GIT_REF={GIT_REF!r}")

SMOKE_TEST=False  REPO_URL='https://github.com/dnwiloic/pfas-gnn.git'  GIT_REF='main'


## Cell 1 — GPU detection & version info

In [2]:
import sys, platform
print(f"Python  : {sys.version}")
print(f"Platform: {platform.platform()}")

# Detect Colab environment
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass
print(f"IN_COLAB: {IN_COLAB}")

if IN_COLAB:
    import shutil, subprocess
    if shutil.which("nvidia-smi") is None:        # CPU runtime: binary absent
        class _R: returncode = 1; stdout = ""
        r = _R()
    else:
        r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if r.returncode == 0:
        print("\n[GPU detected]")
        print(r.stdout[:800])
    else:
        print("[WARNING] No GPU found.")
        print("Note: T2 baselines are CPU-bound (sklearn HGB).")
        print("      A High-RAM CPU instance is often more efficient for this run.")
else:
    print("[Local run — GPU check skipped]")

for pkg_name in ["numpy", "sklearn", "pandas"]:
    try:
        mod = __import__(pkg_name)
        print(f"{pkg_name:12s}: {mod.__version__}")
    except ImportError:
        print(f"{pkg_name:12s}: NOT INSTALLED")

Python  : 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
Platform: Linux-6.17.0-35-generic-x86_64-with-glibc2.39
IN_COLAB: False
[Local run — GPU check skipped]
numpy       : 1.26.4
sklearn     : 1.8.0
pandas      : 2.1.4


## Cell 2 — Install dependencies (Colab only)

PyTorch Geometric is NOT needed for these non-graph baselines.

In [3]:
if IN_COLAB:
    import subprocess, sys
    pkgs = [
        "scikit-learn>=1.4,<2.0",
        "imbalanced-learn>=0.11,<1.0",
        "pyyaml>=6.0,<7.0",
        "pandas>=2.0,<3.0",
        "pyarrow>=14.0",
        "scipy>=1.11",
        "numpy>=1.24",
    ]
    print("Installing packages...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("[ERROR] pip install failed:")
        print(result.stderr[-1000:])
        raise RuntimeError("Dependency installation failed.")
    print("Packages installed. Verifying imports...")
    import imblearn, yaml, pandas, sklearn
    print(f"  sklearn={sklearn.__version__}  imblearn={imblearn.__version__}")
    print("Imports OK.")
else:
    print("[Local run] Skipping pip install — using local environment.")

[Local run] Skipping pip install — using local environment.


## Cell 3 — Bootstrap `src/` and dataset via `git clone`

The repo contains both `src/` (code) and `data/CA-PFAS-ASGWS.parquet` (dataset).
A single `git clone` brings both. **No Google Drive needed.**

**IMPORTANT**: `REPO_URL` and `GIT_REF` must point to an up-to-date remote.
If you have uncommitted local changes, push them first:
```bash
git push origin main
```

In [4]:
import sys, subprocess
from pathlib import Path

if IN_COLAB:
    REPO_LOCAL = Path("/content/pfas-gnn")
    if REPO_LOCAL.exists():
        print(f"Repo already cloned at {REPO_LOCAL} — updating to {GIT_REF} ...")
        subprocess.run(["git", "-C", str(REPO_LOCAL), "fetch", "origin"],
                       capture_output=True, text=True)
        subprocess.run(["git", "-C", str(REPO_LOCAL), "checkout", GIT_REF],
                       capture_output=True, text=True)
        r = subprocess.run(
            ["git", "-C", str(REPO_LOCAL), "reset", "--hard", f"origin/{GIT_REF}"],
            capture_output=True, text=True
        )
        print(r.stdout.strip() or r.stderr.strip())
    else:
        print(f"Cloning {REPO_URL} @ {GIT_REF} ...")
        r = subprocess.run(
            ["git", "clone", "--depth=1", "--branch", GIT_REF, REPO_URL, str(REPO_LOCAL)],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print("[ERROR] git clone failed:")
            print(r.stderr)
            raise RuntimeError(
                f"git clone failed. Check REPO_URL={REPO_URL!r} and GIT_REF={GIT_REF!r}.\n"
                "Ensure the branch exists and the repository is public (or provide auth)."
            )
        print("Cloned OK.")
else:
    REPO_LOCAL = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path("..").resolve()
    print(f"[Local run] Using repo at {REPO_LOCAL}")

repo_str = str(REPO_LOCAL)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
print(f"sys.path[0] = {sys.path[0]}")

[Local run] Using repo at /home/wiloic/M2/Recherche/tmp/pfas/pfas-gnn
sys.path[0] = /home/wiloic/M2/Recherche/tmp/pfas/pfas-gnn


In [5]:
# --- Guard-rail: verify the code is present and up-to-date ---
import sys

for mod in list(sys.modules.keys()):
    if mod.startswith("src"):
        del sys.modules[mod]

try:
    import src.baselines_t2
    import src.baselines_t1
    import src.metrics
except ImportError as e:
    raise ImportError(
        f"CRITICAL: Cannot import src modules: {e}\n"
        f"The clone at {REPO_LOCAL} does not contain src/.\n"
        f"Check REPO_URL={REPO_URL!r} and GIT_REF={GIT_REF!r}."
    )

assert hasattr(src.baselines_t2, "FrequencyClassChain"), (
    "CRITICAL: FrequencyClassChain not found in src.baselines_t2.\n"
    f"The code at {REPO_LOCAL} is OBSOLETE.\n"
    f"Push the latest src/ to the remote (GIT_REF={GIT_REF!r}) then re-run this cell."
)

from src.metrics import REQUIRED
assert set(REQUIRED) == {"roc_auc", "f1", "accuracy", "recall", "precision"}, (
    f"CRITICAL: src.metrics.REQUIRED={REQUIRED} — expected 5 standard metrics. Code may be obsolete."
)

print("[guard-rail] src/ is up-to-date: FrequencyClassChain found, 5 metrics OK.")
import src.baselines_t2 as B
import src.config as C
import src.data as D
import src.splits as S
print("[guard-rail] All src modules imported.")

/home/wiloic/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[guard-rail] src/ is up-to-date: FrequencyClassChain found, 5 metrics OK.
[guard-rail] All src modules imported.


## GPU utilisation check

Confirms whether the GPU is actually exercised. **Only XGBoost uses the GPU** (`device='cuda'`); scikit-learn RandomForest / HistGradientBoosting / LogisticRegression are CPU-only. If no GPU is detected, tree models fall back to CPU automatically.

In [ ]:
# --- GPU utilisation check (XGBoost device='cuda') ---
from src import config as _C
print("GPU available :", _C.gpu_available())
print("XGBoost params:", _C.xgb_device_params())
if _C.gpu_available():
    import xgboost as _xgb, numpy as _np, time as _t
    _X = _np.random.rand(60000, 30); _y = (_np.random.rand(60000) < 0.3).astype(int)
    _t0 = _t.time()
    _xgb.XGBClassifier(n_estimators=80, **_C.xgb_device_params()).fit(_X, _y)
    print(f"XGBoost GPU fit OK in {_t.time()-_t0:.1f}s -> the GPU is being exercised.")
    print("Note: RandomForest / HistGradientBoosting / LogisticRegression stay on CPU (sklearn).")
else:
    print("No GPU detected -> tree models run on CPU.")
    print("For GPU: Runtime > Change runtime type > GPU (XGBoost will then use device='cuda').")


## Cell 4 — Load dataset from cloned repo

The dataset `data/CA-PFAS-ASGWS.parquet` is versioned in the repository and was
downloaded automatically by `git clone` in Cell 3. No Drive mount needed.

Integrity check: shape must be 46338 x 201 with keys `gm_well_id`, `latitude`,
`longitude`, `PFOA_ngL`. Any failure stops execution with an explicit message.

In [6]:
import pandas as pd
from pathlib import Path

# Resolve DATA_PATH relative to the cloned repo
data_file = REPO_LOCAL / DATA_PATH
if not data_file.exists():
    raise FileNotFoundError(
        f"Dataset not found at {data_file}.\n"
        f"Expected path: REPO_LOCAL/DATA_PATH = {REPO_LOCAL}/{DATA_PATH}\n"
        "The parquet file should be versioned in the repo. If missing:\n"
        "  1. Check that the file is committed: git ls-files data/\n"
        "  2. Or update DATA_PATH to its actual location within the repo."
    )

print(f"Dataset: {data_file}  size={data_file.stat().st_size/1e6:.1f} MB")

print("Loading parquet for integrity check...")
_df_check = pd.read_parquet(data_file)
n_rows, n_cols = _df_check.shape
print(f"Shape: {n_rows} x {n_cols}")

REQUIRED_COLS = ["gm_well_id", "latitude", "longitude", "PFOA_ngL"]
missing_cols = [c for c in REQUIRED_COLS if c not in _df_check.columns]
if missing_cols:
    raise ValueError(
        f"INTEGRITY FAIL: columns {missing_cols} missing from the parquet.\n"
        f"File at {data_file} does not match the expected CA-PFAS-ASGWS dataset.\n"
        f"Check DATA_PATH={DATA_PATH!r} or update REPO_URL/GIT_REF."
    )
if n_cols != 201:
    print(f"[WARNING] Expected 201 columns, got {n_cols}. Check dataset version.")
if n_rows != 46338:
    print(f"[WARNING] Expected 46338 rows, got {n_rows}. May be an older snapshot.")
if n_cols == 201 and n_rows == 46338:
    print("[INTEGRITY OK] shape==(46338, 201), required columns present.")
del _df_check

# Override DATA_PARQUET in config
C.DATA_PARQUET = data_file
print(f"src.config.DATA_PARQUET -> {C.DATA_PARQUET}")

Dataset: /home/wiloic/M2/Recherche/tmp/pfas/pfas-gnn/data/CA-PFAS-ASGWS.parquet  size=3.6 MB
Loading parquet for integrity check...
Shape: 46338 x 201
[INTEGRITY OK] shape==(46338, 201), required columns present.
src.config.DATA_PARQUET -> /home/wiloic/M2/Recherche/tmp/pfas/pfas-gnn/data/CA-PFAS-ASGWS.parquet


## Cell 5 — Configure run parameters & experiment directory

Artifacts are written inside the cloned repo workspace (`experiments/<id>/`).
**Warning**: the Colab workspace is ephemeral. See Cell 15 for persistence options.

In [7]:
from pathlib import Path
import numpy as np

# Artifact output directory — inside the cloned repo
EXP_ID   = f"baseline_t2{'_smoke' if SMOKE_TEST else ''}"
SAVE_DIR = REPO_LOCAL / "experiments" / EXP_ID
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifacts will be saved to: {SAVE_DIR}")

# Run configuration
LABELS       = list(C.T2_LABELS)                   # 10 labels (9 core + PFNA)
FEATURE_COLS = C.feature_columns(include_location=False, cocontam="core", include_air=False)
K            = 2 if SMOKE_TEST else 8              # spatial CV folds
SMOKE_N      = 800                                  # wells for smoke mode
SMALL        = SMOKE_TEST                           # small HGB models in smoke
MAX_ITER     = 60 if SMOKE_TEST else 200

print(f"Labels: {LABELS}")
print(f"Features: {len(FEATURE_COLS)} columns")
print(f"K={K}  SMALL={SMALL}  MAX_ITER={MAX_ITER}  SMOKE_N={SMOKE_N if SMOKE_TEST else 'all'}")

Artifacts will be saved to: /home/wiloic/M2/Recherche/tmp/pfas/pfas-gnn/experiments/baseline_t2
Labels: ['PFOS', 'PFBS', 'PFHxA', 'PFOA', 'PFHpA', 'PFBA', 'PFPeA', 'PFHxS', 'PFPeS', 'PFNA']
Features: 53 columns
K=8  SMALL=False  MAX_ITER=200  SMOKE_N=all


## Cell 6 — Load data & build splits

In [8]:
import time

t_load = time.time()
df = D.load(smoke=SMOKE_TEST, smoke_n=SMOKE_N)
print(f"[data] rows={len(df)}  wells={df[C.WELL_ID].nunique()}  "
      f"labels={len(LABELS)}  feats={len(FEATURE_COLS)}  SMOKE={SMOKE_TEST}")

# Verify no feature leakage
leak = set(FEATURE_COLS) & set(C.LEAKAGE_BLOCKLIST)
assert not leak, f"LEAKAGE detected in features: {leak}"
print("[leak check] OK — no PFAS-derived column in feature set.")

# Build folds
spatial = S.spatial_block_folds(df, k=K)
random  = S.group_random_folds(df, k=K)
S.assert_no_group_leak(df, spatial)
S.assert_no_group_leak(df, random)
print(f"[splits] spatial k={len(np.unique(spatial))}  random k={len(np.unique(random))}")

# Measurement mask summary
Y, M = B.masked_targets(df, labels=LABELS)
print("[mask] measured %:")
for a in LABELS:
    pct  = M[f"label_{a}"].mean() * 100
    prev = float(Y[f"label_{a}"].to_numpy()[M[f"label_{a}"].to_numpy()].mean()) * 100
    print(f"  {a:10s}: measured={pct:.1f}%  prevalence={prev:.1f}%")

# Patch max_iter without touching src defaults
_orig_make = B.make_estimator
def _patched_make(kind="hgb", *, class_weight=None, small=False):
    est = _orig_make(kind, class_weight=class_weight, small=small)
    if hasattr(est, "max_iter") and not small:
        est.set_params(max_iter=MAX_ITER)
    return est
B.make_estimator = _patched_make
print(f"[patch] HGB max_iter set to {MAX_ITER} for non-small models.")

[data] rows=46338  wells=11333  labels=10  feats=53  SMOKE=False
[leak check] OK — no PFAS-derived column in feature set.
[splits] spatial k=8  random k=8
[mask] measured %:
  PFOS      : measured=99.8%  prevalence=39.4%
  PFBS      : measured=95.1%  prevalence=39.2%
  PFHxA     : measured=95.8%  prevalence=38.4%
  PFOA      : measured=99.8%  prevalence=34.1%
  PFHpA     : measured=97.3%  prevalence=26.5%
  PFBA      : measured=55.7%  prevalence=41.0%
  PFPeA     : measured=55.5%  prevalence=41.0%
  PFHxS     : measured=95.2%  prevalence=15.4%
  PFPeS     : measured=56.0%  prevalence=15.8%
  PFNA      : measured=97.4%  prevalence=2.6%
[patch] HGB max_iter set to 200 for non-small models.


## Cell 7 — Run 5 models (with per-model checkpointing)

**Smoke** (`SMOKE_TEST=True`): ~800 wells, 2 folds, small models — **CPU < 3 min**.

**Full run** (`SMOKE_TEST=False`): all 46 338 rows, 8 folds, 5 models, inner-CV chains.
Estimated duration: **~60–180 min CPU**. Expected **~30–90 min on Colab High-RAM CPU**.

Per-model checkpointing: `metrics_incremental.json` is updated after each model so that
a Colab disconnection does not lose completed models.

In [ ]:
import json, time

ORDER = tuple(LABELS)
BASE_KIND = B.default_base_kind()  # 'xgb' (GPU) on Colab, 'hgb' (CPU) on smoke
print(f"[compute] GPU={B.C.gpu_available()}  base_learner={BASE_KIND}")

def f_prev():   return B.PrevalenceBaseline(labels=LABELS)
def f_br():     return B.BinaryRelevance(kind=BASE_KIND, labels=LABELS,
                                         class_weight="balanced", small=SMALL)
def f_chain():  return B.MaskedClassifierChain(kind=BASE_KIND, order=ORDER,
                                               out_labels=ORDER, class_weight="balanced",
                                               small=SMALL, inner_k=2 if SMOKE_TEST else 3)
def f_ecc():    return B.EnsembleClassifierChains(kind=BASE_KIND,
                                                  n_chains=2 if SMOKE_TEST else 3,
                                                  labels=list(LABELS),
                                                  class_weight="balanced", small=SMALL)
def f_fcc():    return B.FrequencyClassChain(kind=BASE_KIND, labels=list(LABELS),
                                             n_classes=4, class_weight="balanced",
                                             small=SMALL,
                                             inner_k=2 if SMOKE_TEST else 3)

MODEL_SPECS = [
    ("Prevalence",       f_prev,  False),
    ("BinaryRelevance",  f_br,    False),
    ("Chain",            f_chain, True),
    ("Ensemble",         f_ecc,   True),
    ("FreqClassChain",   f_fcc,   True),
]

results = {}
t_all = time.time()
ckpt_path = SAVE_DIR / "metrics_incremental.json"

for nm, fac, use_groups in MODEL_SPECS:
    t1 = time.time()
    print(f"\n[{nm}] running spatial CV (k={K})...")
    sp = B.evaluate_model(df, fac, spatial, FEATURE_COLS, labels=LABELS,
                          use_groups=use_groups, desc=f"{nm}/spatial")
    print(f"[{nm}] running random CV (k={K})...")
    rd = B.evaluate_model(df, fac, random, FEATURE_COLS, labels=LABELS,
                          use_groups=use_groups, desc=f"{nm}/random")
    results[nm] = {"spatial": sp, "random": rd}
    a, b = sp["aggregate"], rd["aggregate"]
    elapsed_m = time.time() - t1
    print(f"  [{nm}] DONE in {elapsed_m:.0f}s")
    print(f"    spatial: macroAUROC={a['macro_AUROC']:.3f}  microF1={a['micro_F1']:.3f}  "
          f"Hamming={a['Hamming']:.3f}  EMR={a['EMR']:.3f}")
    print(f"    random : macroAUROC={b['macro_AUROC']:.3f}  "
          f"DAUROC={b['macro_AUROC']-a['macro_AUROC']:+.3f}")

    # --- Checkpoint: save incremental results after each model ---
    def _pack_lite(res):
        return {"aggregate": res["aggregate"],
                "thresholds": res["thresholds"],
                "per_label": res["per_label"].to_dict(orient="records"),
                "per_fold": res["per_fold"].to_dict(orient="records"),
                "spread": res["spread"]}
    ckpt = {nm_: {"spatial": _pack_lite(results[nm_]["spatial"]),
                  "random": _pack_lite(results[nm_]["random"])}
            for nm_ in results}
    ckpt_path.write_text(json.dumps(ckpt, indent=2, default=float))
    print(f"  [checkpoint] saved to {ckpt_path}")

print(f"\nAll models done in {time.time()-t_all:.0f}s")


[Prevalence] running spatial CV (k=8)...
[Prevalence] running random CV (k=8)...
  [Prevalence] DONE in 24s
    spatial: macroAUROC=0.348  microF1=0.487  Hamming=0.602  EMR=0.074
    random : macroAUROC=0.471  DAUROC=+0.122
  [checkpoint] saved to /home/wiloic/M2/Recherche/tmp/pfas/pfas-gnn/experiments/baseline_t2/metrics_incremental.json

[BinaryRelevance] running spatial CV (k=8)...
[BinaryRelevance] running random CV (k=8)...
  [BinaryRelevance] DONE in 1987s
    spatial: macroAUROC=0.698  microF1=0.551  Hamming=0.337  EMR=0.156
    random : macroAUROC=0.903  DAUROC=+0.205
  [checkpoint] saved to /home/wiloic/M2/Recherche/tmp/pfas/pfas-gnn/experiments/baseline_t2/metrics_incremental.json

[Chain] running spatial CV (k=8)...


/home/wiloic/.local/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/wiloic/.local/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/wiloic/.local/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/wiloic/.local/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: Us

[Chain] running random CV (k=8)...
  [Chain] DONE in 5815s
    spatial: macroAUROC=0.681  microF1=0.545  Hamming=0.365  EMR=0.137
    random : macroAUROC=0.903  DAUROC=+0.222
  [checkpoint] saved to /home/wiloic/M2/Recherche/tmp/pfas/pfas-gnn/experiments/baseline_t2/metrics_incremental.json

[Ensemble] running spatial CV (k=8)...


## Cell 8 — SMOTE ablation on PFNA (rare label)

In [ ]:
import time

t_smote = time.time()
print("Running SMOTE ablation on PFNA (rare regulated label)...")

def f_br_smote():
    return B.BinaryRelevance(kind=BASE_KIND, labels=LABELS, class_weight="balanced",
                             smote_labels=("PFNA",), small=SMALL)

smote_res = B.evaluate_model(df, f_br_smote, spatial, FEATURE_COLS, labels=LABELS)
pfna_pl_cw = results["BinaryRelevance"]["spatial"]["per_label"]
pfna_cw = float(pfna_pl_cw.loc[pfna_pl_cw.label == "PFNA", "AUROC"].iloc[0])
pfna_sm_pl = smote_res["per_label"]
pfna_sm = float(pfna_sm_pl.loc[pfna_sm_pl.label == "PFNA", "AUROC"].iloc[0])

print(f"PFNA AUROC: class_weight={pfna_cw:.3f}  +SMOTE={pfna_sm:.3f}  "
      f"delta={pfna_sm-pfna_cw:+.3f}  ({time.time()-t_smote:.0f}s)")

## Cell 9 — Pseudo-label probe (semi-supervision on reduced-panel labels)

In [ ]:
import time

t_pseudo = time.time()
print("Running pseudo-label probe (PFBA, PFPeA, PFPeS)...")
probe = B.pseudo_label_probe(df, spatial, FEATURE_COLS,
                             target_labels=("PFBA", "PFPeA", "PFPeS"), small=SMALL)
print(f"Probe done in {time.time()-t_pseudo:.0f}s")
if len(probe):
    print(probe.to_string(index=False, float_format="{:.3f}".format))
else:
    print("(no fold met the size guards in smoke mode -> expected, runs on full data)")

## Cell 10 — Paired comparison BR vs Chain (Wilcoxon)

In [ ]:
import numpy as np

paired = {}
for metric in ["macro_AUROC", "micro_F1"]:
    md, p = B.paired_compare(
        results["BinaryRelevance"]["spatial"]["per_fold"],
        results["Chain"]["spatial"]["per_fold"],
        metric=metric
    )
    paired[metric] = {"chain_minus_br": md, "wilcoxon_p": p}
    sig = "*" if (p is not None and not np.isnan(p) and p < 0.05) else ""
    print(f"[paired] {metric}: chain-BR = {md:+.4f}  Wilcoxon p = {p}{sig}")

## Cell 11 — Results table: models x 5 metrics (spatial + random + Delta)

In [ ]:
import pandas as pd
import numpy as np

print("\n" + "="*90)
print("MODELS x 5 METRICS (micro-averaged) — T2 multilabel baseline")
print("="*90)

rows = []
for nm in ["Prevalence", "BinaryRelevance", "Chain", "Ensemble", "FreqClassChain"]:
    if nm not in results:
        continue
    sp = results[nm]["spatial"]["aggregate"]
    rd = results[nm]["random"]["aggregate"]
    rows.append({
        "model":            nm,
        "macro AUROC (sp)": sp["macro_AUROC"],
        "micro F1 (sp)":    sp["micro_F1"],
        "micro Acc (sp)":   sp["micro_accuracy"],
        "micro Rec (sp)":   sp["micro_recall"],
        "micro Prec (sp)":  sp["micro_precision"],
        "Hamming (sp)":     sp["Hamming"],
        "EMR (sp)":         sp["EMR"],
        "macro AUROC (rd)": rd["macro_AUROC"],
        "D AUROC":          rd["macro_AUROC"] - sp["macro_AUROC"],
    })

df_res = pd.DataFrame(rows).set_index("model")
pd.options.display.float_format = "{:.3f}".format
print(df_res.to_string())

print("\nNote: macro AUROC = unweighted mean over labels (threshold-free).")
print("      micro metrics = pooled over all (row, label) measured cells.")
print("      D AUROC = random - spatial (positive -> spatial autocorrelation artefact).")

## Cell 12 — FrequencyClassChain: 4 frequency classes

In [ ]:
print("FrequencyClassChain — 4 frequency classes (from least rare to rarest)")
print("(Classes determined by prevalence on the smoke-sampled measured rows)\n")

import numpy as np
Yf, Mf = B.masked_targets(df, labels=LABELS)
freq = {a: (float(Yf[f"label_{a}"].to_numpy()[Mf[f"label_{a}"].to_numpy()].mean())
            if Mf[f"label_{a}"].any() else 0.0) for a in LABELS}
order = sorted(LABELS, key=lambda a: -freq[a])
classes = [list(g) for g in np.array_split(np.array(order, dtype=object), 4) if len(g)]

for i, cl in enumerate(classes, 1):
    print(f"  Class {i} ({['least rare','...','...','rarest'][i-1]}): "
          + ", ".join(f"{a} (prev={freq[a]:.3f})" for a in cl))

print()
print("Chain runs in frequency order: each label is predicted from X + all more-frequent")
print("labels already predicted (Dong et al. 2024 cascade approach, leak-free at train).")

## Cell 13 — Per-label results (Binary Relevance vs Chain, spatial CV)

In [ ]:
import pandas as pd

print("PER-LABEL RESULTS — Binary Relevance (spatial CV, OOF thresholds)")
print("5 metrics per label: AUROC / F1 / accuracy / recall / precision\n")

pl_br = results["BinaryRelevance"]["spatial"]["per_label"].copy()
pl_ch = results["Chain"]["spatial"]["per_label"].set_index("label").copy()

pl_br["Chain AUROC"] = [float(pl_ch.loc[a, "AUROC"]) if a in pl_ch.index else float("nan")
                        for a in pl_br["label"]]
pl_br["delta AUROC (Chain-BR)"] = pl_br["Chain AUROC"] - pl_br["AUROC"]

display_cols = ["label", "n_measured", "prevalence",
                "AUROC", "Chain AUROC", "delta AUROC (Chain-BR)",
                "f1", "accuracy", "recall", "precision"]
pl_show = pl_br[[c for c in display_cols if c in pl_br.columns]]
pd.options.display.float_format = "{:.3f}".format
print(pl_show.to_string(index=False))

## Cell 14 — Save all artifacts

**Warning**: the Colab workspace is ephemeral. Run Cell 15 immediately after to persist results.

In [ ]:
import json, yaml, time

t_save = time.time()

# Config
cfg = {
    "task": "T2_multilabel_baseline",
    "seed": int(C.SEED),
    "smoke": SMOKE_TEST,
    "labels": list(LABELS),
    "n_feature_cols": len(FEATURE_COLS),
    "feature_cols": list(FEATURE_COLS),
    "cv": {"spatial_k": int(K), "random_k": int(K), "group_key": C.WELL_ID},
    "models": [nm for nm, _, _ in MODEL_SPECS],
    "hgb_max_iter": MAX_ITER,
    "encode": "frequency",
    "threshold": "per-label F1 on OOF probabilities",
    "imbalance": "class_weight=balanced; SMOTE ablation on PFNA",
}
(SAVE_DIR / "config.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))

# Full metrics
def pack(res):
    return {
        "aggregate":  res["aggregate"],
        "spread":     res["spread"],
        "thresholds": res["thresholds"],
        "per_label":  res["per_label"].to_dict(orient="records"),
        "per_fold":   res["per_fold"].to_dict(orient="records"),
    }

metrics = {
    "smoke": SMOKE_TEST,
    "seed":  int(C.SEED),
    "models": {nm: {"spatial": pack(results[nm]["spatial"]),
                    "random":  pack(results[nm]["random"])}
               for nm in results},
    "smote_ablation_PFNA": {"class_weight": pfna_cw, "smote": pfna_sm},
    "paired_br_vs_chain": paired,
    "pseudo_label_probe": probe.to_dict(orient="records") if len(probe) else [],
}
(SAVE_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, default=float))

# Per-label table as CSV
results["BinaryRelevance"]["spatial"]["per_label"].to_csv(
    SAVE_DIR / "per_label_BR_spatial.csv", index=False
)

print(f"Artifacts saved in {time.time()-t_save:.1f}s to: {SAVE_DIR}")
for f in sorted(SAVE_DIR.iterdir()):
    print(f"  {f.name:45s}  {f.stat().st_size/1024:.1f} KB")

## Cell 15 — Timing summary, full-run estimate & persistence

**WARNING — Colab workspace is ephemeral**: files in `/content/` are lost when the
runtime disconnects. Run the persistence block at the bottom of this cell to either:
- **Option A**: download an archive of `experiments/<id>/` to your local machine.
- **Option B**: commit and push results back to GitHub (requires authentication).

In [ ]:
import time

elapsed_total = time.time() - t_all
print(f"Total run time: {elapsed_total:.1f}s ({elapsed_total/60:.2f} min)")

if SMOKE_TEST:
    n_full = 46338
    n_smoke = len(df)
    scale_rows  = n_full / n_smoke
    scale_folds = 8 / K
    scale_model = 3.5
    lo = elapsed_total * scale_rows * scale_folds * scale_model / 60
    hi = lo * 1.5
    print(f"\nEXTRAPOLATED full run: ~{lo:.0f}-{hi:.0f} min CPU")
    print(f"  (x{scale_rows:.1f} rows, x{scale_folds:.0f} folds, ~x{scale_model} model size)")
    print(f"  On Colab (High-RAM CPU instance): expect ~30-90 min.")
    print()
    print("SMOKE TEST PASSED. To run the full pipeline, set SMOKE_TEST=False.")
    print("Checkpoints saved per model -> Colab disconnection will not lose completed models.")
else:
    print("FULL RUN COMPLETED.")
    print(f"Results in: {SAVE_DIR}")

print()
print("=" * 60)
print("PERSISTENCE — run one of the options below before session ends!")
print("=" * 60)

# --- OPTION A: download archive to your local machine ---
if IN_COLAB:
    import shutil
    from pathlib import Path
    archive_name = f"t2_results_{EXP_ID}"
    archive_path = Path("/content") / archive_name
    shutil.make_archive(str(archive_path), "zip", root_dir=str(SAVE_DIR.parent),
                        base_dir=SAVE_DIR.name)
    zip_file = str(archive_path) + ".zip"
    print(f"Archive created: {zip_file}")
    print("Downloading to browser...")
    try:
        from google.colab import files
        files.download(zip_file)
        print("Download started.")
    except Exception as e:
        print(f"[download error: {e}]")
        print(f"Manually download the archive from: {zip_file}")
else:
    print("[Local run] No download needed — results are at:")
    print(f"  {SAVE_DIR}")

print()
print("--- OPTION B: git push (optional) ---")
print("# Uncomment and run in a new cell if you have GitHub credentials:")
print("# import subprocess")
print("# subprocess.run(['git', '-C', str(REPO_LOCAL), 'add', 'experiments/'], check=True)")
print("# subprocess.run(['git', '-C', str(REPO_LOCAL), 'commit', '-m', 'results: T2 baseline'], check=True)")
print("# subprocess.run(['git', '-C', str(REPO_LOCAL), 'push'], check=True)")